# Camada Bronze

In [0]:
-- Define o catálogo e o schema da camada Bronze
-- que serão utilizados nesta etapa da pipeline.
USE CATALOG mvp;
USE SCHEMA bronze;

In [0]:
-- Remove a tabela caso ela já exista,
-- permitindo sua recriação com os dados consolidados.
DROP TABLE IF EXISTS producao_energetica

In [0]:
%python

# Lista os arquivos CSV armazenados no Volume da camada Staging
# que serão utilizados na consolidação da camada Bronze.
arquivos = dbutils.fs.ls("/Volumes/mvp/staging/dados_anp")

In [0]:
%python
# Cria uma lista vazia para armazenar os DataFrames
# gerados a partir de cada arquivo CSV.
dfs = []

# Percorre todos os arquivos encontrados no Volume.
for arquivo in arquivos:

    # Realiza a leitura de cada CSV como um DataFrame Spark.
    # Considera a primeira linha como cabeçalho
    # e utiliza ponto e vírgula como separador dos campos.
    df = (
        spark.read
             .option("header", True)
             .option("sep", ";")
             .csv(arquivo.path)
    )

    dfs.append(df)

In [0]:
%python
from functools import reduce

# Consolida os DataFrames dos diferentes arquivos CSV
# em um único DataFrame, associando as colunas pelos respectivos nomes.
bronze_df = reduce(
    lambda x, y: x.unionByName(y),
    dfs
)

In [0]:
%python

# Exibe uma amostra dos primeiros 100 registros
# para conferir o resultado da consolidação dos arquivos.
display(bronze_df.limit(100))

Regiao,Unidades da Federacao,Unidade,Produto,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
null,Amazonas,(mil barris),LGN,6.085,6.366,5.794,5.723,6.225,5.995,5.748,5.406,5.099,5.092
null,Ceara,(mil barris),LGN,57,28,-,-,-,-,-,-,-,-
null,Rio Grande do Norte,(mil barris),LGN,1.338,1.144,983,965,849,819,541,527,248,180
null,Alagoas,(mil barris),LGN,516,448,598,502,514,369,389,296,28,547
null,Sergipe,(mil barris),LGN,1.084,899,639,552,454,348,106,-,-,-
null,Bahia,(mil barris),LGN,1.484,1.473,1.397,960,936,880,674,647,766,245
null,Espirito Santo,(mil barris),LGN,6.140,5.382,5.789,5.969,5.476,5.649,5.752,5.097,3.406,3.389
null,Rio de Janeiro,(mil barris),LGN,15.177,14.319,10.043,7.509,5.681,4.330,3.220,3.115,4.774,4.875
null,Sao Paulo,(mil barris),LGN,1.594,2.613,10.164,18.345,19.048,19.309,20.154,18.053,19.147,14.849
null,Amazonas,(milhoes de m3),Gas Natural,"4.703,80","5.060,20","5.106,20","4.756,40","5.216,00","5.571,10","4.957,20","4.957,10","5.067,60","5.213,70"


In [0]:
%python

from pyspark.sql.functions import col

# Cria uma referência do DataFrame consolidado
# para realizar a padronização dos nomes das colunas.
bronze_df_cleaned = bronze_df

# Percorre os cabeçalhos e substitui espaços por "_",
# facilitando a utilização das colunas nas consultas posteriores.
for column in bronze_df.columns:
    clean_name = column.replace(" ", "_")
    bronze_df_cleaned = bronze_df_cleaned.withColumnRenamed(column, clean_name)

# Persiste o DataFrame consolidado no formato Delta
# como a tabela producao_energetica da camada Bronze.
# O modo overwrite substitui a tabela caso ela já exista.
bronze_df_cleaned.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("mvp.bronze.producao_energetica")

* Conferênca do resultado final da tabela **_mvp.bronze.producao_energetica_**

In [0]:
-- Consulta a tabela Bronze para validar os dados
-- consolidados e persistidos após o processo de ingestão.

SELECT * 
FROM mvp.bronze.producao_energetica

Regiao,Unidades_da_Federacao,Unidade,Produto,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
Regiao Norte,Rondonia,(mil m3),Etanol anidro e hidratado,"12,77","12,99","9,06","4,9","1,39","4,85","0,07",-,-,-
Regiao Norte,Acre,(mil m3),Etanol anidro e hidratado,-,"4,51","3,67",-,-,-,-,-,-,-
Regiao Norte,Amazonas,(mil m3),Etanol anidro e hidratado,"2,92","5,8","5,5","4,85","5,47","8,82","9,01","6,39","5,9","6,81"
Regiao Norte,Para,(mil m3),Etanol anidro e hidratado,"42,15","40,93","33,15","51,62","43,46","61,23","50,34","54,98","52,77","54,13"
Regiao Norte,Tocantins,(mil m3),Etanol anidro e hidratado,"180,72","189,81","161,97","176,27","155,22","166,37","174,5","195,01","209,58","192,71"
Regiao Nordeste,Maranhao,(mil m3),Etanol anidro e hidratado,"179,15","186,98",128,"162,56","147,62","167,74","174,55","164,33","143,11","160,69"
Regiao Nordeste,Piaui,(mil m3),Etanol anidro e hidratado,"32,51","32,68","21,61","20,4","37,48","46,46","38,61","44,14","45,03","47,16"
Regiao Nordeste,Ceara,(mil m3),Etanol anidro e hidratado,"9,13","14,6","5,24",-,-,-,-,-,-,-
Regiao Nordeste,Rio Grande do Norte,(mil m3),Etanol anidro e hidratado,"73,24","98,26","75,15","66,35","114,9","109,64","118,3","101,78","91,81","138,28"
Regiao Nordeste,Paraiba,(mil m3),Etanol anidro e hidratado,"375,7","447,06","360,23","329,63","431,04","359,03","395,53","362,88","361,86","419,27"
